# Lost in the Middle — Llama 3.1 8B | 30 Documents | Remaining Positions
**Model:** llama3.1:8b | Positions: **9, 14, 19, 24, 29** | **30 docs**  
**GPU:** 2x T4 | Estimated time: ~20 hours  

This notebook evaluates Llama 3.1 8B on the remaining positions for the 30-document setting. 
It uses the **improved, paper-exact evaluation metric**:
- First-line truncation (`output.split('\n')[0].strip()`)
- Checking against all valid answer synonyms (`any(normalize(ans) in prediction for ans in target_answers)`)
- Saving every 50 questions incrementally to support easy resuming.

In [ ]:
!pip install ollama

In [ ]:
import subprocess
import time

print("Installing dependencies...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run("pip install ollama==0.6.2", shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Starting server...")
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

print("Pulling model...")
subprocess.run("ollama pull llama3.1:8b", shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Ollama ready!")


In [ ]:
import os
import json
import string
import re
import time
import ollama

# --- CONFIG ---
MODEL_NAME = "llama3.1:8b"
NUM_CTX = 12288   # Safe headroom for 30 docs
DATASET_PATH = "/kaggle/input/datasets/arinjaysarkar/qa-questions"
OUTPUT_DIR = "/kaggle/working"

positions_to_run = [9, 14, 19, 24, 29]
folder = "30_total_documents"

def normalize_answer(s: str) -> str:
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

# Files to store intermediate progress
responses_file = os.path.join(OUTPUT_DIR, "all_responses_llama3_30docs_remaining.json")
results_file = os.path.join(OUTPUT_DIR, "results_llama3_30docs_remaining.json")
dataset_responses_file = "/kaggle/input/datasets/unknownarunkownar/all-responses/all_responses_llama3_30docs_remaining.json"

# Load existing progress if available
if os.path.exists(responses_file):
    with open(responses_file, "r") as f:
        all_responses = json.load(f)
    print(f"Loaded {len(all_responses)} existing responses from /kaggle/working/.")
elif os.path.exists(dataset_responses_file):
    with open(dataset_responses_file, "r") as f:
        all_responses = json.load(f)
    print(f"Loaded {len(all_responses)} existing responses from Kaggle input dataset.")
else:
    all_responses = []
    print("No existing responses found. Starting fresh.")
if os.path.exists(results_file):
    with open(results_file, "r") as f:
        results_data = json.load(f)
else:
    results_data = []

base_path = os.path.join(DATASET_PATH, "qa_data", folder)
total_start = time.time()

for pos in positions_to_run:
    # Check if this position is already fully completed
    completed_for_pos = [r for r in all_responses if r["position"] == pos]
    if len(completed_for_pos) == 2655:
        print(f"Position {pos} is already completed. Skipping.")
        continue
    
    filename = f"nq-open-{folder}_gold_at_{pos}.jsonl"
    filepath = os.path.join(base_path, filename)
    
    # Load dataset lines
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()
        total_questions = len(lines)
    
    # Count how many we already processed for this position
    start_q = len(completed_for_pos)
    correct_matches = sum(1 for r in completed_for_pos if r["match"])
    
    print(f"\n{'='*40}")
    print(f">>> Position {pos} | Resuming from Q{start_q} | Correct so far: {correct_matches} <<<")
    print(f"{'='*40}")
    
    pos_start = time.time()
    
    for q_idx in range(start_q, total_questions):
        data = json.loads(lines[q_idx])
        target_query = data["question"]
        target_answers = data["answers"]
        ctxs = data["ctxs"]
        
        # Format context
        context_parts = []
        for j, ctx in enumerate(ctxs):
            context_parts.append(f"Document [{j+1}](Title: {ctx['title']}) {ctx['text']}")
        context_str = "\n".join(context_parts)
        
        prompt = (
            "Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant).\n\n"
            f"{context_str}\n\n"
            f"Question: {target_query}\n"
            "Answer: (Keep the answer as concise as possible, preferably one or two words. Do not write full sentences.)"
        )
        
        try:
            q_start = time.time()
            response = ollama.chat(
                model=MODEL_NAME,
                messages=[{'role': 'user', 'content': prompt}],
                options={'num_ctx': NUM_CTX, 'temperature': 0.0}
            )
            q_time = time.time() - q_start
            raw_output = response['message']['content']
            
            # Paper-exact metric: Truncate at first newline
            output = raw_output.split("\n")[0].strip()
            normalized_prediction = normalize_answer(output)
            
            success = any(
                normalize_answer(ans) in normalized_prediction
                for ans in target_answers
            )
            if success:
                correct_matches += 1
                
            all_responses.append({
                "folder": folder,
                "position": pos,
                "q_idx": q_idx,
                "question": target_query,
                "target_answer": target_answers[0],
                "model_response": raw_output.strip(),
                "match": success,
                "time_sec": round(q_time, 2)
            })
            
            # Save progress every 50 questions
            if (q_idx + 1) % 50 == 0 or q_idx == total_questions - 1:
                with open(responses_file, "w", encoding="utf-8") as f:
                    json.dump(all_responses, f, indent=2)
                
                accuracy_so_far = (correct_matches / (q_idx + 1)) * 100
                elapsed = time.time() - pos_start
                eta = (elapsed / (q_idx - start_q + 1)) * (total_questions - q_idx - 1) if (q_idx - start_q + 1) > 0 else 0
                print(f"[Q{q_idx+1}/{total_questions}] ({q_time:.1f}s) "
                      f"Acc: {accuracy_so_far:.1f}% | "
                      f"ETA: {eta/60:.0f}min | "
                      f"{target_answers[0]} -> {output[:50]} | {success}")
                      
        except Exception as e:
            print(f"Error at Q{q_idx+1}: {str(e)}")
            all_responses.append({
                "folder": folder,
                "position": pos,
                "q_idx": q_idx,
                "question": target_query,
                "target_answer": target_answers[0],
                "model_response": f"ERROR: {str(e)}",
                "match": False,
                "time_sec": 0
            })
            
    # Position complete
    accuracy = (correct_matches / total_questions) * 100
    pos_time = time.time() - pos_start
    print(f"\n>> Position {pos} DONE: {accuracy:.1f}% ({correct_matches}/{total_questions}) [{pos_time/60:.1f}min]")
    
    # Update results summary
    results_data = [r for r in results_data if r["Position of Answer (Gold Index)"] != pos]
    results_data.append({
        "Context Size (Documents)": "30",
        "Position of Answer (Gold Index)": pos,
        "Accuracy (%)": round(accuracy, 2),
        "Correct": correct_matches,
        "Total": total_questions
    })
    
    with open(results_file, "w", encoding="utf-8") as f:
        json.dump(results_data, f, indent=2)

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"ALL DONE! Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*60}")